# Custom Attention Kernels

## Historical Context

Attention has been the bottleneck in transformers since 2017:

### Timeline
- **2017**: "Attention is All You Need" (Vaswani et al.)
  - Quadratic memory and compute: O(N^2)
  - Limits sequence length to ~512-2048 tokens

- **2019-2020**: Efficient Attention Variants
  - **Reformer** (Kitaev et al., 2020): LSH attention
  - **Linformer** (Wang et al., 2020): Linear projections
  - **Performer** (Choromanski et al., 2020): Random features
  - Problem: Approximations lose quality

- **2021**: Sparse Attention
  - **Longformer** (Beltagy et al., 2020): Windowed + global
  - **BigBird** (Zaheer et al., 2020): Random + window + global
  - Limited speedup due to irregular memory access

- **2022**: Flash Attention (Breakthrough)
  - **Flash Attention** (Dao et al., June 2022)
  - Key insight: IO-awareness, not algorithm changes
  - **Exact** attention, 2-4x faster, 10-20x less memory
  - Enabled training with 16K+ context

- **2023**: Flash Attention 2
  - Better parallelization across threads/warps
  - Work partitioning optimizations
  - 2x faster than Flash Attention 1

- **2024**: Variants and Extensions
  - **Flash Decoding**: Optimized for inference
  - **Paged Attention** (vLLM): KV cache management
  - **Multi-Query/Grouped-Query Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Optional, Tuple
from dataclasses import dataclass

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

# Check for Flash Attention
try:
    from flash_attn import flash_attn_func
    FLASH_ATTN_AVAILABLE = True
    print("Flash Attention available")
except ImportError:
    FLASH_ATTN_AVAILABLE = False
    print("Flash Attention not available (install: pip install flash-attn)")

## Standard Attention: The Baseline

### Algorithm
```
1. Compute scores: S = Q @ K^T / sqrt(d)     [B, H, N, N]
2. Apply mask (optional)
3. Softmax: P = softmax(S, dim=-1)          [B, H, N, N]
4. Weighted sum: O = P @ V                   [B, H, N, D]
```

### Problems
1. **Memory**: O(N^2) for attention matrix
   - For N=4096, head_dim=64, batch=1, heads=8:
   - Attention matrix: 1 * 8 * 4096 * 4096 * 4 bytes = 512 MB
   - For N=16384: 8 GB just for attention!

2. **Memory Bandwidth**: Multiple HBM round trips
   - Load Q, K -> compute S -> store S to HBM
   - Load S -> softmax -> store P to HBM  
   - Load P, V -> compute O
   - Total: 4 reads + 3 writes = 7 HBM accesses

3. **GPU Underutilization**: Memory-bound, not compute-bound

In [ ]:
def standard_attention(
    q: torch.Tensor,  # [batch, heads, seq_len, head_dim]
    k: torch.Tensor,
    v: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
    dropout_p: float = 0.0,
) -> torch.Tensor:
    """
    Standard scaled dot-product attention.
    
    This is the textbook implementation that materializes
    the full attention matrix.
    """
    # Get dimensions
    batch, heads, seq_len, head_dim = q.shape
    
    # Scale factor
    scale = head_dim ** -0.5
    
    # Compute attention scores [B, H, N, N]
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    
    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax [B, H, N, N]
    attn_weights = F.softmax(scores, dim=-1)
    
    # Apply dropout
    if dropout_p > 0.0:
        attn_weights = F.dropout(attn_weights, p=dropout_p)
    
    # Weighted sum [B, H, N, D]
    output = torch.matmul(attn_weights, v)
    
    return output, attn_weights


# Test standard attention
if torch.cuda.is_available():
    batch, heads, seq_len, head_dim = 2, 8, 512, 64
    
    q = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    
    # Measure memory
    torch.cuda.reset_peak_memory_stats()
    output, attn_weights = standard_attention(q, k, v)
    memory_used = torch.cuda.max_memory_allocated() / 1e9
    
    print(f"Input shape: {q.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Attention weights shape: {attn_weights.shape}")
    print(f"\nMemory used: {memory_used:.3f} GB")
    
    # Calculate theoretical attention matrix size
    attn_matrix_size = batch * heads * seq_len * seq_len * 4 / 1e9  # 4 bytes for float32
    print(f"Attention matrix size: {attn_matrix_size:.3f} GB")
    
    # Verify attention weights sum to 1
    print(f"\nAttention weights sum (should be 1.0): {attn_weights.sum(dim=-1).mean():.6f}")

## Memory Analysis: Why Standard Attention is Slow

Let's visualize memory requirements and bandwidth.

In [ ]:
# Memory analysis for different sequence lengths
def analyze_memory_requirements():
    """Calculate memory requirements for standard attention."""
    seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
    batch = 1
    heads = 8
    head_dim = 64
    bytes_per_element = 4  # float32
    
    results = []
    
    for N in seq_lengths:
        # Input/output memory
        qkv_memory = 3 * batch * heads * N * head_dim * bytes_per_element
        output_memory = batch * heads * N * head_dim * bytes_per_element
        
        # Attention matrix memory (the killer)
        attn_matrix_memory = batch * heads * N * N * bytes_per_element
        
        # Total
        total_memory = qkv_memory + output_memory + attn_matrix_memory
        
        results.append({
            'seq_len': N,
            'qkv_gb': qkv_memory / 1e9,
            'attn_gb': attn_matrix_memory / 1e9,
            'total_gb': total_memory / 1e9,
        })
    
    return results

results = analyze_memory_requirements()

# Print table
print("Memory Requirements for Standard Attention (batch=1, heads=8, dim=64)\n")
print(f"{'Seq Len':<10} {'Q/K/V (GB)':<15} {'Attention (GB)':<15} {'Total (GB)':<15}")
print("-" * 60)

for r in results:
    print(f"{r['seq_len']:<10} {r['qkv_gb']:<15.3f} {r['attn_gb']:<15.3f} {r['total_gb']:<15.3f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

seq_lens = [r['seq_len'] for r in results]
qkv_mem = [r['qkv_gb'] for r in results]
attn_mem = [r['attn_gb'] for r in results]
total_mem = [r['total_gb'] for r in results]

# Stacked bar chart
ax1.bar(range(len(seq_lens)), qkv_mem, label='Q/K/V/O', alpha=0.8)
ax1.bar(range(len(seq_lens)), attn_mem, bottom=qkv_mem, label='Attention Matrix', alpha=0.8)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Memory (GB)')
ax1.set_title('Memory Breakdown: Standard Attention')
ax1.set_xticks(range(len(seq_lens)))
ax1.set_xticklabels(seq_lens, rotation=45)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Log scale to show O(N^2) growth
ax2.plot(seq_lens, attn_mem, 'o-', linewidth=2, markersize=8, label='Attention Matrix')
ax2.plot(seq_lens, qkv_mem, 's-', linewidth=2, markersize=8, label='Q/K/V/O')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Memory (GB, log scale)')
ax2.set_title('Memory Scaling')
ax2.set_xscale('log', base=2)
ax2.set_yscale('log')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('attention_memory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insight: Attention matrix grows O(N^2), dominates memory at long sequences")
print("For N=16384: Attention matrix alone requires 16+ GB!")

## Flash Attention: The Solution

### Key Insights (Dao et al., 2022)

1. **IO-Awareness**: Modern GPUs are memory-bound, not compute-bound
   - A100: 312 TFLOPS compute, 1.5 TB/s memory bandwidth
   - Attention is limited by memory, not compute

2. **Tiling**: Never materialize full attention matrix
   - Process in blocks that fit in SRAM (shared memory)
   - Compute partial attention, aggregate results

3. **Recomputation**: Trade compute for memory
   - Backward pass: recompute attention on-the-fly
   - Still faster due to reduced memory traffic

4. **Online Softmax**: Numerically stable incremental softmax
   - Update running max and sum as we process tiles
   - No need to see full row before computing softmax

### Algorithm Overview
```
For each block of queries Q_i:
    Initialize output O_i, running max m_i, running sum l_i
    
    For each block of keys/values K_j, V_j:
        Load Q_i, K_j, V_j into SRAM
        Compute S_ij = Q_i @ K_j^T
        
        # Online softmax update
        Update m_i with max(m_i, max(S_ij))
        Compute P_ij = exp(S_ij - m_i)
        Update l_i and O_i
    
    Final normalization: O_i = O_i / l_i
```

### Complexity
- **Memory**: O(N) vs O(N^2)
- **HBM accesses**: O(N^2 d^2 / M) vs O(N^2 d)
  - M = SRAM size, typically d << M << N*d
- **FLOPs**: Same as standard (but better utilization)

In [ ]:
def online_softmax_demo():
    """
    Demonstrate online softmax algorithm.
    
    Key insight: Can compute softmax incrementally without
    seeing all values at once.
    """
    # Example: compute softmax over [1, 2, 3, 4, 5]
    # Process in two blocks: [1, 2, 3] and [4, 5]
    
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    
    # Standard softmax (for verification)
    standard = F.softmax(x, dim=0)
    
    # Online softmax
    # Block 1: [1, 2, 3]
    x1 = torch.tensor([1.0, 2.0, 3.0])
    m1 = x1.max()
    d1 = torch.exp(x1 - m1)
    l1 = d1.sum()
    
    print(f"Block 1: {x1}")
    print(f"  max: {m1}, sum_exp: {l1}")
    
    # Block 2: [4, 5]
    x2 = torch.tensor([4.0, 5.0])
    m2 = x2.max()
    
    print(f"\nBlock 2: {x2}")
    print(f"  max: {m2}")
    
    # Update global max and rescale
    m_global = max(m1, m2)
    print(f"\nGlobal max: {m_global}")
    
    # Rescale block 1 results
    correction1 = torch.exp(m1 - m_global)
    d1_corrected = d1 * correction1
    l1_corrected = l1 * correction1
    
    # Process block 2
    d2 = torch.exp(x2 - m_global)
    l2 = d2.sum()
    
    # Combine
    l_global = l1_corrected + l2
    online = torch.cat([d1_corrected, d2]) / l_global
    
    print(f"\nStandard softmax: {standard}")
    print(f"Online softmax:   {online}")
    print(f"Max difference: {(standard - online).abs().max():.2e}")
    
    return standard, online

online_softmax_demo()

## Flash Attention Implementation

Using PyTorch's built-in scaled_dot_product_attention (uses Flash Attention 2 internally).

In [ ]:
def flash_attention_pytorch(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
    dropout_p: float = 0.0,
) -> torch.Tensor:
    """
    Flash Attention using PyTorch's optimized implementation.
    
    PyTorch 2.0+ includes Flash Attention 2 for compute capability >= 7.5
    Falls back to efficient implementations on other hardware.
    """
    # Convert mask format if needed (PyTorch expects additive mask)
    attn_mask = None
    if mask is not None:
        attn_mask = torch.zeros_like(mask, dtype=q.dtype)
        attn_mask.masked_fill_(mask == 0, float('-inf'))
    
    # Use PyTorch's optimized attention
    output = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attn_mask,
        dropout_p=dropout_p,
        is_causal=False,
    )
    
    return output


# Test Flash Attention
if torch.cuda.is_available():
    batch, heads, seq_len, head_dim = 2, 8, 2048, 64
    
    q = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    
    # Flash Attention
    torch.cuda.reset_peak_memory_stats()
    output_flash = flash_attention_pytorch(q, k, v)
    memory_flash = torch.cuda.max_memory_allocated() / 1e9
    
    # Standard attention (for comparison)
    torch.cuda.reset_peak_memory_stats()
    output_std, _ = standard_attention(q, k, v)
    memory_std = torch.cuda.max_memory_allocated() / 1e9
    
    print(f"Sequence length: {seq_len}")
    print(f"\nMemory Usage:")
    print(f"  Standard: {memory_std:.3f} GB")
    print(f"  Flash:    {memory_flash:.3f} GB")
    print(f"  Reduction: {memory_std / memory_flash:.2f}x")
    
    print(f"\nNumerical Accuracy:")
    print(f"  Max diff: {(output_flash - output_std).abs().max():.2e}")
    print(f"  Mean diff: {(output_flash - output_std).abs().mean():.2e}")

## Comprehensive Benchmark: Standard vs Flash Attention

In [ ]:
def benchmark_attention_implementations(seq_len, batch=2, heads=8, head_dim=64, n_iter=50):
    """
    Comprehensive benchmark of attention implementations.
    """
    q = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float16)
    k = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float16)
    v = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float16)
    
    results = {}
    
    # Benchmark standard attention
    try:
        # Warmup
        for _ in range(5):
            output, _ = standard_attention(q, k, v)
        torch.cuda.synchronize()
        
        # Measure
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        for _ in range(n_iter):
            output, _ = standard_attention(q, k, v)
        torch.cuda.synchronize()
        
        time_std = (time.time() - start) / n_iter * 1000
        memory_std = torch.cuda.max_memory_allocated() / 1e9
        
        results['standard'] = {
            'time': time_std,
            'memory': memory_std,
            'output': output
        }
    except RuntimeError as e:
        if 'out of memory' in str(e):
            results['standard'] = {'time': None, 'memory': None, 'output': None, 'oom': True}
            torch.cuda.empty_cache()
        else:
            raise
    
    # Benchmark Flash Attention (PyTorch)
    # Warmup
    for _ in range(5):
        output = flash_attention_pytorch(q, k, v)
    torch.cuda.synchronize()
    
    # Measure
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    for _ in range(n_iter):
        output = flash_attention_pytorch(q, k, v)
    torch.cuda.synchronize()
    
    time_flash = (time.time() - start) / n_iter * 1000
    memory_flash = torch.cuda.max_memory_allocated() / 1e9
    
    results['flash_pytorch'] = {
        'time': time_flash,
        'memory': memory_flash,
        'output': output
    }
    
    # Benchmark flash-attn library if available
    if FLASH_ATTN_AVAILABLE:
        # Rearrange to [batch, seq_len, heads, head_dim] for flash-attn
        q_fa = q.transpose(1, 2)
        k_fa = k.transpose(1, 2)
        v_fa = v.transpose(1, 2)
        
        # Warmup
        for _ in range(5):
            output = flash_attn_func(q_fa, k_fa, v_fa)
        torch.cuda.synchronize()
        
        # Measure
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        for _ in range(n_iter):
            output = flash_attn_func(q_fa, k_fa, v_fa)
        torch.cuda.synchronize()
        
        time_flash_lib = (time.time() - start) / n_iter * 1000
        memory_flash_lib = torch.cuda.max_memory_allocated() / 1e9
        
        results['flash_attn_lib'] = {
            'time': time_flash_lib,
            'memory': memory_flash_lib,
            'output': output.transpose(1, 2)
        }
    
    return results


if torch.cuda.is_available():
    # Benchmark different sequence lengths
    seq_lengths = [512, 1024, 2048, 4096, 8192]
    
    benchmark_results = []
    
    print("Attention Implementation Benchmark\n")
    print(f"{'Seq Len':<10} {'Impl':<20} {'Time (ms)':<12} {'Memory (GB)':<12} {'Speedup':<10}")
    print("-" * 70)
    
    for seq_len in seq_lengths:
        results = benchmark_attention_implementations(seq_len)
        
        # Get baseline (Flash if standard OOM)
        if results['standard'].get('oom', False):
            baseline_time = results['flash_pytorch']['time']
            print(f"{seq_len:<10} {'Standard':<20} {'OOM':<12} {'OOM':<12} {'-':<10}")
        else:
            baseline_time = results['standard']['time']
            print(f"{seq_len:<10} {'Standard':<20} {results['standard']['time']:<12.2f} "
                  f"{results['standard']['memory']:<12.3f} {'1.00x':<10}")
        
        # Flash PyTorch
        speedup = baseline_time / results['flash_pytorch']['time']
        print(f"{seq_len:<10} {'Flash (PyTorch)':<20} {results['flash_pytorch']['time']:<12.2f} "
              f"{results['flash_pytorch']['memory']:<12.3f} {speedup:<10.2f}x")
        
        # Flash-attn library
        if 'flash_attn_lib' in results:
            speedup = baseline_time / results['flash_attn_lib']['time']
            print(f"{seq_len:<10} {'Flash (flash-attn)':<20} {results['flash_attn_lib']['time']:<12.2f} "
                  f"{results['flash_attn_lib']['memory']:<12.3f} {speedup:<10.2f}x")
        
        print()
        
        benchmark_results.append({
            'seq_len': seq_len,
            'results': results
        })

In [ ]:
if torch.cuda.is_available() and benchmark_results:
    # Visualize benchmark results
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    seq_lens = []
    std_times, flash_pt_times, flash_lib_times = [], [], []
    std_mems, flash_pt_mems, flash_lib_mems = [], [], []
    
    for item in benchmark_results:
        seq_len = item['seq_len']
        results = item['results']
        
        seq_lens.append(seq_len)
        
        if not results['standard'].get('oom', False):
            std_times.append(results['standard']['time'])
            std_mems.append(results['standard']['memory'])
        else:
            std_times.append(None)
            std_mems.append(None)
        
        flash_pt_times.append(results['flash_pytorch']['time'])
        flash_pt_mems.append(results['flash_pytorch']['memory'])
        
        if 'flash_attn_lib' in results:
            flash_lib_times.append(results['flash_attn_lib']['time'])
            flash_lib_mems.append(results['flash_attn_lib']['memory'])
        else:
            flash_lib_times.append(None)
            flash_lib_mems.append(None)
    
    # Time comparison
    if any(t is not None for t in std_times):
        valid_idx = [i for i, t in enumerate(std_times) if t is not None]
        axes[0, 0].plot([seq_lens[i] for i in valid_idx], [std_times[i] for i in valid_idx], 
                       'o-', label='Standard', linewidth=2)
    axes[0, 0].plot(seq_lens, flash_pt_times, 's-', label='Flash (PyTorch)', linewidth=2)
    if any(t is not None for t in flash_lib_times):
        valid_idx = [i for i, t in enumerate(flash_lib_times) if t is not None]
        axes[0, 0].plot([seq_lens[i] for i in valid_idx], [flash_lib_times[i] for i in valid_idx],
                       '^-', label='Flash (flash-attn)', linewidth=2)
    axes[0, 0].set_xlabel('Sequence Length')
    axes[0, 0].set_ylabel('Time (ms)')
    axes[0, 0].set_title('Attention Time Comparison')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].set_xscale('log', base=2)
    
    # Speedup
    speedups = []
    for i, seq_len in enumerate(seq_lens):
        if std_times[i] is not None:
            speedups.append(std_times[i] / flash_pt_times[i])
        else:
            speedups.append(None)
    
    valid_idx = [i for i, s in enumerate(speedups) if s is not None]
    if valid_idx:
        axes[0, 1].plot([seq_lens[i] for i in valid_idx], [speedups[i] for i in valid_idx],
                       'o-', linewidth=2, markersize=8)
        axes[0, 1].axhline(y=1, color='r', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Sequence Length')
    axes[0, 1].set_ylabel('Speedup (Standard / Flash)')
    axes[0, 1].set_title('Flash Attention Speedup')
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].set_xscale('log', base=2)
    
    # Memory comparison
    if any(m is not None for m in std_mems):
        valid_idx = [i for i, m in enumerate(std_mems) if m is not None]
        axes[1, 0].plot([seq_lens[i] for i in valid_idx], [std_mems[i] for i in valid_idx],
                       'o-', label='Standard', linewidth=2)
    axes[1, 0].plot(seq_lens, flash_pt_mems, 's-', label='Flash (PyTorch)', linewidth=2)
    axes[1, 0].set_xlabel('Sequence Length')
    axes[1, 0].set_ylabel('Memory (GB)')
    axes[1, 0].set_title('Memory Usage Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].set_xscale('log', base=2)
    
    # Memory reduction
    mem_reductions = []
    for i in range(len(seq_lens)):
        if std_mems[i] is not None:
            mem_reductions.append(1 - flash_pt_mems[i] / std_mems[i])
        else:
            mem_reductions.append(None)
    
    valid_idx = [i for i, m in enumerate(mem_reductions) if m is not None]
    if valid_idx:
        axes[1, 1].bar([seq_lens[i] for i in valid_idx], [mem_reductions[i] for i in valid_idx], alpha=0.8)
    axes[1, 1].set_xlabel('Sequence Length')
    axes[1, 1].set_ylabel('Memory Reduction')
    axes[1, 1].set_title('Flash Attention Memory Savings')
    axes[1, 1].set_xscale('log', base=2)
    axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('flash_attention_comprehensive_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()

## Causal (Autoregressive) Attention

For language models, we need causal masking (can't attend to future tokens).

In [ ]:
def create_causal_mask(seq_len, device='cuda'):
    """Create causal mask for autoregressive attention."""
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    return mask == 0  # True where we can attend

def causal_attention_standard(q, k, v):
    """Standard causal attention."""
    seq_len = q.size(2)
    mask = create_causal_mask(seq_len, device=q.device)
    return standard_attention(q, k, v, mask=mask)

def causal_attention_flash(q, k, v):
    """Flash causal attention (more efficient)."""
    output = F.scaled_dot_product_attention(
        q, k, v,
        is_causal=True,  # More efficient than explicit mask
    )
    return output


if torch.cuda.is_available():
    # Test causal attention
    batch, heads, seq_len, head_dim = 2, 8, 1024, 64
    
    q = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    k = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    v = torch.randn(batch, heads, seq_len, head_dim, device='cuda')
    
    # Standard
    output_std, attn_std = causal_attention_standard(q, k, v)
    
    # Flash
    output_flash = causal_attention_flash(q, k, v)
    
    print(f"Causal Attention Test (seq_len={seq_len})")
    print(f"\nMax diff: {(output_std - output_flash).abs().max():.2e}")
    print(f"Mean diff: {(output_std - output_flash).abs().mean():.2e}")
    
    # Visualize attention pattern
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Show first head of first example
    attn_pattern = attn_std[0, 0, :50, :50].cpu().numpy()
    
    im1 = axes[0].imshow(attn_pattern, cmap='viridis', aspect='auto')
    axes[0].set_title('Causal Attention Pattern')
    axes[0].set_xlabel('Key Position')
    axes[0].set_ylabel('Query Position')
    plt.colorbar(im1, ax=axes[0])
    
    # Show mask
    mask = create_causal_mask(50).cpu().numpy()
    im2 = axes[1].imshow(mask, cmap='binary', aspect='auto')
    axes[1].set_title('Causal Mask')
    axes[1].set_xlabel('Key Position')
    axes[1].set_ylabel('Query Position')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    plt.savefig('causal_attention_pattern.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nCausal mask ensures each position only attends to past (including itself)")

## Summary

### Key Insights

1. **Standard Attention Bottleneck**
   - O(N^2) memory for attention matrix
   - 7 HBM accesses per attention operation
   - Memory-bound, not compute-bound

2. **Flash Attention Solution**
   - Tiling + online softmax
   - O(N) memory complexity
   - 2-4x faster, 10-20x less memory
   - Exact (not approximate!)

3. **Practical Impact**
   - Enables 16K-128K context windows
   - Essential for long-context models
   - Now standard in PyTorch, Hugging Face

4. **When to Use What**
   - Always prefer Flash Attention (PyTorch 2.0+)
   - Use `F.scaled_dot_product_attention`
   - Falls back to efficient implementation automatically

### Performance Summary

For seq_len=4096, batch=1, heads=8, dim=64:
- **Standard**: ~512 MB memory, ~15 ms
- **Flash**: ~50 MB memory, ~6 ms
- **Speedup**: 2.5x faster, 10x less memory

### Next Steps

- **Notebook 53**: Fused operations (LayerNorm + residual, etc.)
- **Notebook 54**: Quantization methods

### Resources

- [Flash Attention Paper](https://arxiv.org/abs/2205.14135)
- [Flash Attention 2 Paper](https://arxiv.org/abs/2307.08691)
- [flash-attn Library](https://github.com/Dao-AILab/flash-attention)
- [PyTorch SDPA Docs](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)